In [ ]:
try:
    from google.colab import drive
    drive.mount('/gdrive')
    dataset_root = '/gdrive/MyDrive/datasets'
    !pip install torchinfo tqdm
    colab = True
except Exception as e:
    print(e)
    print('Assuming we\'re not on colab.')
    dataset_root = './datasets'
    colab = False

print('Will store datasets in', dataset_root)

import os

if os.name == 'nt':
    print("Disabling multiprocessing because we're running on windows.")
    cpu_num = 0
elif colab:
    cpu_num = 2
else:
    cpu_num = os.cpu_count() // 2
    print('Dataloaders will use {} CPUs'.format(cpu_num))

In [ ]:
import torch
import torch.utils.data as tud
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

import torchvision.datasets as tds
import torchvision.utils as tu

import matplotlib.pyplot as plt

In [ ]:
# https://github.com/zalandoresearch/fashion-mnist
mnist_train = tds.FashionMNIST(
    root=dataset_root,
    download=True,
    train=True,
)

mnist_eval = tds.FashionMNIST(
    root=dataset_root,
    download=True,
    train=False,
)

As a first exercise and introduction to tensors, let's see what an "average" T-shirt looks like.

In [ ]:
shirts = mnist_train.data[mnist_train.targets == 0]
print(shirts.shape)
sample = shirts[0]
avg_shirt = shirts.float().mean(dim=0)
print(avg_shirt.shape)
plt.imshow(torch.cat([avg_shirt, sample], dim=1))

In [ ]:
class MnistBinary:
    def __init__(self, mnist: tds.FashionMNIST, label0: int, label1: int):
        d0 = mnist.data[mnist.targets == label0]
        d1 = mnist.data[mnist.targets == label1]

        l0 = torch.zeros(len(d0))
        l1 = torch.ones(len(d1))
        
        self.data = torch.cat((d0, d1))
        self.targets = torch.cat((l0, l1))

    def __getitem__(self, index: int):
        return (self.data[index], self.targets[index])

    def __len__(self):
        return len(self.data)

    def random_grid(self, sz: int):
        samples_ix = torch.randint(low=0, high=len(self.data) - 1, size=(sz,))
        samples = self.data[samples_ix]
        samples = samples[:, None, ...]
        grid = tu.make_grid(samples)
        return grid.permute(1, 2, 0)        

In [ ]:
# The classes to use are selected here: 0 for T-shirts, 5 for Sandals
train = MnistBinary(mnist_train, 0, 5)
val = MnistBinary(mnist_eval, 0, 5)
plt.imshow(val.random_grid(64))

In [ ]:
batchsize = 32

train_loader = tud.DataLoader(train, batch_size=batchsize, num_workers=cpu_num, shuffle=True)
val_loader = tud.DataLoader(val, batch_size=batchsize, shuffle=True)

In [ ]:
class MyNn(nn.Module):
    def __init__(self, in_sz, out_sz, hidden_sz=128):
        super().__init__()
        self.in_sz = in_sz
        self.out_sz = out_sz
        self.lin1 = nn.Linear(in_sz, hidden_sz)
        self.lin2 = nn.Linear(hidden_sz, out_sz)

    def forward(self, x):
        x = x.view(-1, self.in_sz)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)
        x = F.sigmoid(x)
        return x
        

In [ ]:
testmodel = MyNn(28 * 28, 1)
print(summary(testmodel))
test = train[0][0].unsqueeze(0).float()
testmodel(test)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MyNn(28 * 28, 1, hidden_sz=8).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
lossfn = nn.L1Loss()
epochs = 10

for epoch in range(epochs):
    for i, (images, target) in enumerate(train_loader):
        optimizer.zero_grad()
        images = images.float().to(device)
        targets = target.float().to(device)

        outs = model(images)
        loss = lossfn(outs, targets.view(-1, 1))
        loss.backward()
        optimizer.step()

    losses = []
    for i, (images, target) in enumerate(val_loader):
        with torch.no_grad():
            images = images.float().to(device)
            targets = target.float().to(device)
            outs = model(images)
            loss = lossfn(outs, targets.view(-1, 1))
            losses.append(loss)
    print("Val loss {}".format(torch.Tensor(losses).mean()))

In [ ]:
total = len(val)
correct = 0
with torch.no_grad():
    for image, target in val:
        pred = model(image.float().to(device))
        if (pred.round() == target):
            correct += 1
            
    print("{:.2f}% correct".format(100*correct/total))

In [ ]:
from ipywidgets import interact

@interact(index=(0, len(val) - 1, 1))
def draw_preds(index=0):
    with torch.no_grad():
        image = val[index][0]
        pred = model(image.float().to(device))
        # pred = pred, dim=1)
        plt.imshow(image.float().cpu().squeeze(), cmap='gray')
        print(pred, pred.round())

How does such a simple neural network score >99% correct on binary classification? Well...

In [ ]:
with torch.no_grad():
    test = torch.zeros(28,28)
    test[3:5, 8:10] = 255
    plt.imshow(test, cmap='gray')
    pred = model(test.float().to(device))
    print(pred)

A 2x2 white square in the upper left quadrant is good enough to be classified as a shirt. If you look at the training images, the majority of the sandals never have any white pixels in this region. The NN is able to exploit this fact, and confidently predict shirts without learning a more complicated idea of what a shirt is.

What would our score be if we make a "model" that classifies based on those four pixels?

In [ ]:
total = len(val)
correct = 0
with torch.no_grad():
    for image, target in val:
        if image[3:5, 8:10].sum() > 10:
            pred = 0
        else:
            pred = 1
        if (pred == target):
            correct += 1
            
    print("{:.2f}% correct".format(100*correct/total))

As homework, try train the model on different pairs of classes rather than shoes and shirts. You can see the other options in the Fashion MNIST page here https://github.com/zalandoresearch/fashion-mnist.

What was the final score for your chosen pair? Visually, do you think your pair should be easy or difficult to discriminate?

Finally, repeat the 4-pixel experiment with your model trained on a different class pair. Can you find a small pixel region that reaches similar classification accuracy to your model?